# Notebook 01: Data Pull and Inspection

Pull Statcast pitch-level data for a selected pitcher and inspect the raw structure.

This notebook is a **thin consumer** of the data layer (`src/data/`). No analytical logic lives here.

In [ ]:
import sys
sys.path.insert(0, '..')  # Make src importable from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from src.services.pitcher_service import get_pitcher_dataset, get_available_games, get_available_pitch_types
from src import config

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)
%matplotlib inline

## Parameters

Edit these to select a different pitcher or date range.

In [ ]:
# ---- Edit these ----
PITCHER = "Gerrit Cole"          # Name string OR integer MLBAM ID
START_DATE = "2024-04-01"
END_DATE   = "2024-09-30"
USE_CACHE  = True                # Set False to force re-fetch
# --------------------

## Pull data

First call fetches from pybaseball and caches locally. Subsequent calls load from cache.

In [ ]:
df = get_pitcher_dataset(
    pitcher_id=PITCHER,
    start_date=START_DATE,
    end_date=END_DATE,
    use_cache=USE_CACHE,
)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Available games

In [ ]:
games = get_available_games(df)
print(f"{len(games)} games loaded:")
for g in games:
    print(f"  {g}")

## Pitch type distribution

In [ ]:
print("Pitch type counts across all loaded games:")
print(df[config.PITCH_TYPE_COL].value_counts())

df[config.PITCH_TYPE_COL].value_counts().plot(
    kind='bar', color='steelblue', edgecolor='black', figsize=(8, 4)
)
plt.title(f"{PITCHER} — Pitch Mix ({START_DATE} to {END_DATE})")
plt.xlabel("Pitch type")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Location overview: all pitch types

In [ ]:
from src.visualization.zone_plots import draw_strike_zone

fig, ax = plt.subplots(figsize=(6, 7))
ax.scatter(
    df[config.PLATE_X_COL], df[config.PLATE_Z_COL],
    alpha=0.08, s=6, color='gray'
)
draw_strike_zone(ax)
ax.set_xlim(config.PLOT_X_MIN, config.PLOT_X_MAX)
ax.set_ylim(config.PLOT_Z_MIN, config.PLOT_Z_MAX)
ax.set_aspect('equal')
ax.set_title(f"{PITCHER} — All pitch locations ({START_DATE} to {END_DATE})")
ax.set_xlabel("plate_x (ft)")
ax.set_ylabel("plate_z (ft)")
plt.tight_layout()
plt.show()

## Data quality checks

In [ ]:
print("Null counts per column:")
print(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

print(f"\nplate_x range: {df['plate_x'].min():.2f} to {df['plate_x'].max():.2f}")
print(f"plate_z range: {df['plate_z'].min():.2f} to {df['plate_z'].max():.2f}")
print(f"\nGames: {df['game_pk'].nunique() if 'game_pk' in df.columns else 'N/A'}")
print(f"game_pitch_index present: {'game_pitch_index' in df.columns}")
print(f"game_bucket present:       {'game_bucket' in df.columns}")